In [1]:
import sys
import importlib

sys.path.append('../utils')
import visualization
importlib.reload(visualization)
from visualization import save_to_html

import pandas as pd
import plotly.express as px

df = pd.read_parquet('../data/features/ml_dataset.parquet')
display(df.shape)
df.info()
df.head()

(16200, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16200 entries, 0 to 16199
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   sku                     16200 non-null  string  
 1   year                    16200 non-null  int32   
 2   month                   16200 non-null  int32   
 3   qty                     16200 non-null  int64   
 4   revenue                 16200 non-null  float64 
 5   is_fulfilled_by_amazon  16200 non-null  int8    
 6   category                16200 non-null  category
 7   size                    16200 non-null  category
 8   stock_sum               16200 non-null  float64 
 9   in_stock                16200 non-null  int8    
 10  stock_log               16200 non-null  float64 
 11  category_avg_qty        16200 non-null  float64 
dtypes: category(2), float64(4), int32(2), int64(1), int8(2), string(1)
memory usage: 950.1 KB


,sku,year,month,qty,revenue,is_fulfilled_by_amazon,category,size,stock_sum,in_stock,stock_log,category_avg_qty
0,AN201-RED-M,2022,4,1,229.0,1,Bottom,M,5.0,1,1.791759,1.433213
1,AN201-RED-M,2022,5,1,229.0,0,Bottom,M,5.0,1,1.791759,1.433213
2,AN201-RED-XL,2022,6,2,602.0,1,Bottom,XL,6.0,1,1.945910,1.433213
3,AN201-RED-XXL,2022,5,1,229.0,0,Bottom,XXL,3.0,1,1.386294,1.433213
4,AN202-ORANGE-M,2022,4,1,229.0,0,Bottom,M,3.0,1,1.386294,1.433213


In [ ]:
# Basic EDA

# Словари графиков и таблиц для сохранения в html-файл
eda_figs = {}
eda_tables = {}

describe_qty = (pd.DataFrame(df['qty']
                            .describe())
                            .reset_index()
                            .rename(columns={'index': 'metric'})
                            )
eda_tables['Описание qty'] = describe_qty

print('Описание qty:')
display(describe_qty)

fig = px.histogram(
    df,
    x='qty',
    nbins=400,
    range_x=[0, 400],
    title='Distribution of quantity sold'
)
eda_figs['Distribution of quantity sold'] = fig
fig.show()

Описание qty:


,metric,qty
0,count,16200.000000
1,mean,7.190062
2,std,14.382634
3,min,1.000000
4,25%,2.000000
5,50%,3.000000
6,75%,7.000000
7,max,376.000000


In [3]:
describe_revenue = (pd.DataFrame(df['revenue']
                            .describe())
                            .reset_index()
                            .rename(columns={'index': 'metric'})
                            )
eda_tables['Описание revenue'] = describe_revenue

print('Описание revenue:')
display(describe_revenue)

fig = px.histogram(
    df,
    x='revenue',
    nbins=50,
    range_x=[0, 280000],
    title='Revenue distribution'
)
eda_figs['Revenue distribution'] = fig
fig.show()

Описание revenue:


,metric,revenue
0,count,16200.000000
1,mean,4654.313025
2,std,10190.318806
3,min,0.000000
4,25%,818.750000
5,50%,1836.000000
6,75%,4596.750000
7,max,274978.000000


In [4]:
describe_stock_sum = (pd.DataFrame(df['stock_sum']
                            .describe())
                            .reset_index()
                            .rename(columns={'index': 'metric'})
                            )
eda_tables['Описание stock_sum'] = describe_stock_sum

print('Описание stock_sum:')
display(describe_stock_sum)

fig = px.histogram(
    df,
    x='stock_sum',
    nbins=50,
    range_x=[0, 1500],
    title='Stock distribution'
)
eda_figs['Stock distribution'] = fig
fig.show()

Описание stock_sum:


,metric,stock_sum
0,count,16200.000000
1,mean,30.807778
2,std,70.684645
3,min,0.000000
4,25%,3.000000
5,50%,9.000000
6,75%,33.000000
7,max,1234.000000


In [5]:
monthly_sales = (
    df
    .groupby(['year', 'month'], observed=True)['revenue']
    .sum()
    .reset_index() 
)

fig = px.line(
    monthly_sales,
    x='month',
    y='revenue',
    color='year',  
    title='Monthly revenue',
    markers=True, 
    labels={
        'month': 'Month',
        'revenue': 'Revenue',
        'year': 'Year'
    }
)
eda_figs['Monthly revenue'] = fig
fig.show()

In [6]:
df_plot = df.groupby('is_fulfilled_by_amazon')['revenue'].mean().reset_index()
df_plot = df_plot.rename(columns={'revenue': 'revenue_mean'})
display(df_plot)

fig = px.bar(
    df_plot,
    x='is_fulfilled_by_amazon',
    y='revenue_mean',
    title='Average revenue by Amazon fulfillment',
    labels={
        'is_fulfilled_by_amazon': 'Fulfilled by Amazon',
        'revenue_mean': 'Average Revenue'
    },
    text_auto='.2f'  
)
eda_figs['Average revenue by Amazon fulfillment'] = fig
fig.show()

,is_fulfilled_by_amazon,revenue_mean
0,0,1588.005160
1,1,5271.121302


In [7]:
df_plot = df.groupby('in_stock')['qty'].mean().reset_index()
df_plot = df_plot.rename(columns={'qty': 'qty_mean'})

print(df_plot)

fig = px.bar(
    df_plot,
    x='in_stock',
    y='qty_mean',
    title='Average quantity by stock status',
    labels={
        'in_stock': 'In Stock',
        'qty_mean': 'Average Quantity'
    },
    text_auto='.2f'  
)
eda_figs['Average quantity by stock status'] = fig
fig.show()

   in_stock  qty_mean
0         0  5.794905
1         1  7.335674


In [ ]:
# ABC-анализ по SKU

# Словари графиков и таблиц для сохранения в html-файл
abc_figs = {}
abc_tables = {}

top_sku = (
    df.groupby('sku', observed=True)['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

top_sku['cum_share'] = top_sku['revenue'].cumsum() / top_sku['revenue'].sum()
top_sku['cum_share_pct'] = top_sku['cum_share'] * 100

top_sku['ABC'] = pd.cut(
    top_sku['cum_share'],
    bins=[0, 0.80, 0.95, 1.0],
    labels=['A', 'B', 'C'],
    include_lowest=True
)

abc_summary = top_sku.groupby('ABC', observed=True).agg(
    SKUs=('sku', 'count'),
    SKUs_Pct=('sku', lambda x: x.count()/len(top_sku)*100),
    Revenue=('revenue', 'sum'),
    Revenue_Pct=('revenue', lambda x: x.sum()/top_sku['revenue'].sum()*100)
).round(2).reset_index()

abc_tables['ABC-анализ по SKU'] = abc_summary

print("ABC-анализ по SKU:")
display(abc_summary)

fig = px.line(top_sku, x=top_sku.index, y='cum_share_pct',
              title='Pareto Analysis: Кумулятивная доля выручки по SKU',
              labels={'x': 'Количество SKU (по убыванию выручки)', 
                      'cum_share_pct': 'Кумулятивная доля выручки (%)'})

fig.add_hline(y=80, line_dash="dash", line_color="green", annotation_text="80% выручки")
fig.add_hline(y=95, line_dash="dash", line_color="orange", annotation_text="95% выручки")
fig.update_layout(height=600, template="plotly_white")

abc_figs['Pareto Analysis'] = fig
fig.show()

ABC-анализ по SKU:


,ABC,SKUs,SKUs_Pct,Revenue,Revenue_Pct
0,A,1969,27.61,60318091.0,80.0
1,B,2142,30.03,11310743.0,15.0
2,C,3021,42.36,3771037.0,5.0


In [9]:
# ABC-анализ по Category

abc_category = (
    df.groupby('category', observed=True)
    .agg(
        Revenue=('revenue', 'sum'),
        Qty=('qty', 'sum'),
        Orders=('sku', 'nunique'),
        Avg_Price=('revenue', lambda x: x.sum() / x.count())
    )
    .round(2)
    .sort_values('Revenue', ascending=False)
    .reset_index()
)

abc_category['Cum_Revenue'] = abc_category['Revenue'].cumsum()
abc_category['Cum_Pct'] = abc_category['Cum_Revenue'] / abc_category['Revenue'].sum() * 100

abc_category['ABC_Category'] = pd.cut(
    abc_category['Cum_Pct'],
    bins=[0, 80, 95, 101],
    labels=['A', 'B', 'C'],
    right=False
)

abc_tables['ABC-анализ по категориям'] = abc_category

print("ABC-анализ по категориям")
display(abc_category)

fig = px.bar(abc_category, x='category', y='Revenue', 
              color='ABC_Category', title='ABC-анализ по категориям (по выручке)')
fig.add_scatter(x=abc_category['category'], y=abc_category['Cum_Pct'], 
                 mode='lines', name='Кумулятивная доля %', yaxis='y2')
fig.update_layout(
    yaxis2=dict(title='Кумулятивная доля (%)', overlaying='y', side='right'),
    legend=dict(x=1.1)
    )
abc_figs['ABC-анализ по категориям'] = fig
fig.show()

ABC-анализ по категориям


,category,Revenue,Qty,Orders,Avg_Price,Cum_Revenue,Cum_Pct,ABC_Category
0,Set,37660322.0,45223,2478,6341.19,37660322.0,49.947462,A
1,kurta,20451608.0,44969,2901,3109.56,58111930.0,77.071657,A
2,Western Dress,10629096.0,13939,455,9601.71,68741026.0,91.168625,B
3,Top,5203733.0,9899,784,3025.43,73944759.0,98.070140,C
4,Ethnic Dress,760711.0,1053,92,3710.79,74705470.0,99.079042,C
5,Blouse,434751.0,844,149,1622.21,75140221.0,99.655636,C
6,Bottom,140226.0,397,195,506.23,75280447.0,99.841612,C
7,Saree,118509.0,152,75,1139.51,75398956.0,99.998786,C
8,Dupatta,915.0,3,3,305.00,75399871.0,100.000000,C


In [10]:
# ABC-анализ по Size 

abc_size = (
    df.groupby('size', observed=True)
    .agg(
        Revenue=('revenue', 'sum'),
        Qty=('qty', 'sum'),
        Orders=('sku', 'nunique')
    )
    .round(2)
    .sort_values('Revenue', ascending=False)
    .reset_index()
)
abc_size['Cum_Revenue'] = abc_size['Revenue'].cumsum()
abc_size['Cum_Pct'] = abc_size['Cum_Revenue'] / abc_size['Revenue'].sum() * 100


abc_size['ABC_Size'] = pd.cut(
    abc_size['Cum_Pct'],
    bins=[0, 80, 95, 101],
    labels=['A', 'B', 'C'],
    right=False
)
abc_tables['ABC-анализ по размерам'] = abc_size


print("ABC-анализ по размерам")
display(abc_size)

fig = px.bar(abc_size, x='size', y='Revenue', 
              color='ABC_Size', title='ABC-анализ по размеру (по выручке)')
fig.add_scatter(x=abc_size['size'], y=abc_category['Cum_Pct'], 
                 mode='lines', name='Кумулятивная доля %', yaxis='y2')
fig.update_layout(
    yaxis2=dict(title='Кумулятивная доля (%)', overlaying='y', side='right'),
    legend=dict(x=1.1)
    )
abc_figs['ABC-анализ по размерам'] = fig
fig.show()

ABC-анализ по размерам


,size,Revenue,Qty,Orders,Cum_Revenue,Cum_Pct,ABC_Size
0,M,13301382.0,20426,999,13301382.0,17.641120,A
1,L,12684744.0,19972,986,25986126.0,34.464417,A
2,XL,11916874.0,18900,1016,37903000.0,50.269317,A
3,XXL,10258118.0,16490,1030,48161118.0,63.874271,A
4,S,10186895.0,15313,1060,58348013.0,77.384765,A
5,3XL,8794922.0,13505,890,67142935.0,89.049138,B
6,XS,6759023.0,9929,945,73901958.0,98.013375,C
7,6XL,562718.0,687,27,74464676.0,98.759686,C
8,5XL,414809.0,512,28,74879485.0,99.309832,C
9,4XL,325318.0,398,28,75204803.0,99.741289,C


In [11]:
# ABC по паре (Category + Size)

abc_cat_size = (
    df.groupby(['category', 'size'], observed=True)
    .agg(
        Revenue=('revenue', 'sum'),
        Qty=('qty', 'sum')
    )
    .round(2)
    .sort_values('Revenue', ascending=False)
    .reset_index()
)
abc_cat_size['Cum_Pct'] = abc_cat_size['Revenue'].cumsum() / abc_cat_size['Revenue'].sum() * 100
abc_cat_size['ABC'] = pd.cut(
    abc_cat_size['Cum_Pct'],
    bins=[0, 80, 95, 100],
    labels=['A', 'B', 'C']
)
abc_tables['Топ-15 самых важных комбинаций Category + Size'] = abc_cat_size.head(15)

print("\nТоп-15 самых важных комбинаций Category + Size")
display(abc_cat_size.head(15))

abc_cat_size['ABC'] = abc_cat_size['ABC'].astype(str)
fig = px.treemap(abc_cat_size, 
                  path=['category', 'size'], 
                  values='Revenue',
                  color='ABC',
                  title='Treemap: Выручка по Category + Size (цвет = ABC-класс)')
abc_figs['Treemap: Выручка по Category + Size'] = fig
fig.show()


Топ-15 самых важных комбинаций Category + Size


,category,size,Revenue,Qty,Cum_Pct,ABC
0,Set,M,6975134.0,8348,9.250857,A
1,Set,L,6104693.0,7394,17.347280,A
2,Set,XL,5679725.0,6859,24.880085,A
3,Set,S,5621797.0,6697,32.336062,A
4,Set,XXL,4606076.0,5598,38.444927,A
5,Set,3XL,4350547.0,5279,44.214893,A
6,Set,XS,4109002.0,4873,49.664507,A
7,kurta,L,3579090.0,8021,54.411319,A
8,kurta,XL,3473041.0,7830,59.017482,A
9,kurta,M,3419517.0,7701,63.552658,A


In [12]:
num_cols = ['qty','revenue','stock_sum','stock_log', 'category_avg_qty']

corr_df = df[num_cols].corr().reset_index().rename(columns={'index': '/'})
eda_tables['Корреляции числовых признаков'] = corr_df
print('Корреляции числовых признаков')
display(corr_df)


Корреляции числовых признаков


,/,qty,revenue,stock_sum,stock_log,category_avg_qty
0,qty,1.000000,0.934263,0.241563,0.143464,0.128901
1,revenue,0.934263,1.000000,0.181684,0.123679,0.176158
2,stock_sum,0.241563,0.181684,1.000000,0.632399,-0.013401
3,stock_log,0.143464,0.123679,0.632399,1.000000,-0.011422
4,category_avg_qty,0.128901,0.176158,-0.013401,-0.011422,1.000000


In [13]:
df.isna().sum().sort_values(ascending=False)

sku                       0
year                      0
month                     0
qty                       0
revenue                   0
is_fulfilled_by_amazon    0
category                  0
size                      0
stock_sum                 0
in_stock                  0
stock_log                 0
category_avg_qty          0
dtype: int64

In [ ]:
# Сохранение графиков и таблиц в html-файл

save_to_html(
    figures_dict=eda_figs, 
    tables_dict=eda_tables, 
    filename="basic_eda.html", 
    title="Базовый EDA", 
    dashboard_dir="eda")

save_to_html(
    figures_dict=abc_figs, 
    tables_dict=abc_tables, 
    filename="abc_analysis.html", 
    title="ABC-анализ", 
    dashboard_dir="eda")

'Сохранено в ../dashboards/eda/abc_analysis.html'

### **Выводы по Exploratory Data Analysis**

#### **Общая характеристика данных**

За период с марта по июнь 2022 года было продано товаров на общую сумму **75 399 871 INR**.  
Всего реализовано **116 479 единиц** продукции.

Данные демонстрируют высокую концентрацию продаж как по товарным категориям, так и по объёму стока.

#### **Ключевые инсайты**

**1. Высокая концентрация выручки по категориям**

- Категория **Set** является безусловным лидером и приносит **49,95%** всей выручки.
- Две категории — **Set** и **kurta** — совместно обеспечивают **77,07%** от общего оборота.
- Топ-3 категорий (**Set**, **kurta**, **Western Dress**) аккумулируют **91,17%** выручки.

ABC-анализ подтвердил классический принцип Парето: небольшое количество категорий формирует основную часть дохода.

**2. Влияние стока на продажи**

- Средний запас на SKU составляет 30,8 единиц при медиане всего 9 единиц, что указывает на сильное смещение распределения.
- Товары в наличии продаются в среднем на **26,6%** лучше, чем товары с нулевым стоком.
- Признаки стока (`stock_sum` и `stock_log`) показали наибольшую важность в последующем моделировании, подтвердив, что наличие товара на складе является одним из ключевых драйверов продаж.

**3. Эффективность фулфилмента**

- Заказы, выполненные через **Amazon Fulfillment**, имеют средний чек **5271 INR**, что более чем в **3 раза выше**, чем у Merchant (1588 INR).
- Это указывает на значительное преимущество использования инфраструктуры Amazon.

**4. Сезонность продаж**

- Максимальная выручка зафиксирована в **апреле** (27,58 млн INR).
- В мае и июне наблюдается постепенное снижение продаж.

**5. ABC-анализ**

- Анализ по категориям показал ярко выраженное разделение: класс A формирует более 77% выручки.
- Анализ по комбинации `category + size` подтвердил, что определённые размеры (особенно L, XL и XXL) внутри топ-категорий значительно превосходят остальные по вкладу в выручку.
- Pareto-анализ по SKU дополнительно подтвердил высокую концентрацию продаж среди небольшого числа товаров.

#### **Итоговый вывод**

EDA выявил сильную концентрацию выручки в категориях **Set** и **kurta**, а также высокую зависимость продаж от наличия и объёма стока. 

Использование Amazon Fulfillment даёт существенное преимущество по среднему чеку. Полученные инсайты подчёркивают важность грамотного управления ассортиментом и складскими остатками и будут использованы при построении предсказательной модели спроса.